# Stage 03 - Prepare hydro, solar, and wind availability

This notebook creates hourly `p_max_pu` profiles for every hydro, solar, and onshore-wind plant in the 2022 SNI fleet. It validates exact plant coverage, 8,760-hour alignment, finite values, and the physical range from zero to one.

These are availability constraints, not simulated generation. Annual energy and the 5% carrier gate are evaluated only after the network is solved in Stage 06.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.artifacts import format_preflight_report, run_preflight
from src.environment import check_environment, format_environment_report

environment = check_environment()
if not environment.ok:
    raise RuntimeError(format_environment_report(environment))
preflight = run_preflight("03_profiles", root=ROOT)
preflight.raise_for_errors()
print(format_environment_report(environment))
print(format_preflight_report(preflight))

## Hydrology method and evidence limits

The retained source has eight monthly flow proxy series. Each run-of-river plant uses its assigned series divided by its 95th percentile and clipped to `[0, 1]`. Reservoir plants first use a trailing three-month mean. Monthly values are anchored near mid-month and interpolated hourly, with edge values extended to the calendar boundaries.

All 65 hydro plants have an explicit assignment, but most are legacy regional proxies rather than measured plant inflows. Assignment basis and confidence are versioned instead of treating all manual aliases as high confidence. Sarapullo and Sabanilla are forced to zero before their official 2022 operating dates.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.availability import (
    attach_renewable_availability,
    build_hydro_availability,
    load_hydro_assignments,
    load_monthly_flows,
)

generators = pd.read_csv(ROOT / "data/processed/generation/generators_2022.csv")
flows = load_monthly_flows(ROOT / "data/raw/generation/caudals.csv")
assignments = load_hydro_assignments(ROOT / "data/interim/generation/hydro_profile_assignments_2022.csv")
hydro, hydro_diagnostics = build_hydro_availability(generators, flows, assignments, year=2022)
hydro_diagnostics.groupby("confidence", as_index=False).agg(
    plants=("plant_id", "size"),
    effective_capacity_mw=("effective_capacity_mw", "sum"),
)

In [ ]:
figure_dir = ROOT / "reports/figures/phase_04"
figure_dir.mkdir(parents=True, exist_ok=True)
confidence_order = ["high", "medium", "low"]
confidence = hydro_diagnostics.groupby("confidence")["effective_capacity_mw"].sum().reindex(confidence_order)
ax = confidence.plot.bar(figsize=(7.5, 4.2), color=["#2a9d8f", "#e9c46a", "#e76f51"])
ax.set_ylabel("Effective hydro capacity (MW)")
ax.set_xlabel("Assignment confidence")
ax.set_title("Hydro capacity by flow-assignment confidence")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(figure_dir / "hydro_assignment_confidence.png", dpi=160)
plt.show()

## Solar and onshore-wind weather profiles

The external NetCDF files contain 2013 ERA5-derived provincial profiles. Phase 4 does not claim that 2013 weather is observed 2022 weather. It preserves this known weather year as a transparent meteorological scenario, maps each plant by province, and transfers the 8,760 hourly positions to the 2022 non-leap calendar. Galapagos remains outside the SNI plant fleet. Huascachaca is forced to zero before its first official December 2022 unit-group date; the conflicting staged event and plant capacities remain separate evidence rather than an invented fractional trajectory.

In [ ]:
renewable, renewable_diagnostics = attach_renewable_availability(
    generators,
    solar_profile_path=ROOT / "data/raw/cutouts/solar.nc",
    wind_profile_path=ROOT / "data/raw/cutouts/onwind.nc",
    target_year=2022,
)
renewable_diagnostics.groupby("carrier", as_index=False).agg(
    plants=("plant_id", "size"),
    mean_p_max_pu=("mean_p_max_pu", "mean"),
    maximum_p_max_pu=("maximum_p_max_pu", "max"),
)

## Representative one-week profiles

The four panels below show the first seven days of 2022 for one plant of each requested profile type. Selection is deterministic: the largest full-year solar plant, the largest full-year wind plant, the largest reservoir hydro plant, and the largest run-of-river hydro plant by effective capacity. Full-year renewable plants are used so the January window is not dominated by an in-year commissioning mask. All panels share the same time axis and use the same `[0, 1]` availability scale.

In [ ]:
week = slice("2022-01-01 00:00:00", "2022-01-07 23:00:00")
full_year = generators[generators["operating_start_date_2022"] == "2022-01-01"]
solar_example = full_year[full_year["carrier"] == "solar"].sort_values(["p_nom_mw", "plant_id"], ascending=[False, True]).iloc[0]
wind_example = full_year[full_year["carrier"] == "onwind"].sort_values(["p_nom_mw", "plant_id"], ascending=[False, True]).iloc[0]
reservoir_example = hydro_diagnostics[hydro_diagnostics["hydro_model"] == "reservoir_3_month_rolling"].sort_values(["effective_capacity_mw", "plant_id"], ascending=[False, True]).iloc[0]
run_of_river_example = hydro_diagnostics[hydro_diagnostics["hydro_model"] == "run_of_river"].sort_values(["effective_capacity_mw", "plant_id"], ascending=[False, True]).iloc[0]

examples = [
    (renewable[solar_example["plant_id"]], f"Solar - {solar_example['name']}", "#e9c46a"),
    (renewable[wind_example["plant_id"]], f"Wind - {wind_example['name']}", "#2a9d8f"),
    (hydro[reservoir_example["plant_id"]], f"Hydro reservoir - {reservoir_example['plant_name']}", "#457b9d"),
    (hydro[run_of_river_example["plant_id"]], f"Hydro run-of-river - {run_of_river_example['plant_name']}", "#1d3557"),
]
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for axis, (series, title, color) in zip(axes, examples):
    series.loc[week].plot(ax=axis, color=color, linewidth=1.25)
    axis.set_title(title, loc="left")
    axis.set_ylabel("p.u.")
    axis.set_ylim(0, 1.02)
    axis.grid(axis="y", alpha=0.25)
axes[-1].set_xlabel("2022 timestamp (Ecuador local standard time)")
fig.suptitle("Representative one-week availability profiles", y=0.995)
plt.tight_layout()
plt.savefig(figure_dir / "weekly_example_availability_profiles.png", dpi=160, bbox_inches="tight")
plt.show()
pd.DataFrame({
    "profile_type": ["solar", "wind", "hydro_reservoir", "hydro_run_of_river"],
    "plant": [solar_example["name"], wind_example["name"], reservoir_example["plant_name"], run_of_river_example["plant_name"]],
})

## Profile diagnostics and outputs

Duration curves reveal the range and shape of the constraints without implying generation dispatch. The large hourly CSVs are reproducible products and are intentionally ignored by Git; downstream stages regenerate them through this notebook.

In [ ]:
duration = pd.DataFrame({
    "hydro_mean": hydro.mean(axis=1).sort_values(ascending=False).to_numpy(),
    "solar_mean": renewable.loc[:, renewable_diagnostics.query("carrier == 'solar'")["plant_id"]].mean(axis=1).sort_values(ascending=False).to_numpy(),
    "wind_mean": renewable.loc[:, renewable_diagnostics.query("carrier == 'onwind'")["plant_id"]].mean(axis=1).sort_values(ascending=False).to_numpy(),
})
duration.index = 100 * duration.index / (len(duration) - 1)
ax = duration.plot(figsize=(8, 4.5))
ax.set_xlabel("Hours exceeded (%)")
ax.set_ylabel("Mean availability (p.u.)")
ax.set_title("2022-model availability duration curves")
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.savefig(figure_dir / "availability_duration_curves.png", dpi=160)
plt.show()

In [ ]:
output_dir = ROOT / "data/processed/generation"
hydro.to_csv(output_dir / "hydro_availability_2022.csv", index_label="timestamp")
renewable.to_csv(output_dir / "renewable_availability_2022.csv", index_label="timestamp")

summary = {
    "hydro_plants": hydro.shape[1],
    "solar_plants": int((renewable_diagnostics["carrier"] == "solar").sum()),
    "wind_plants": int((renewable_diagnostics["carrier"] == "onwind").sum()),
    "snapshots": hydro.shape[0],
    "hydro_minimum_pu": float(hydro.min().min()),
    "hydro_maximum_pu": float(hydro.max().max()),
    "renewable_minimum_pu": float(renewable.min().min()),
    "renewable_maximum_pu": float(renewable.max().max()),
    "missing_values": int(hydro.isna().sum().sum() + renewable.isna().sum().sum()),
    "weather_year": int(renewable_diagnostics["weather_year"].iloc[0]),
    "simulation_year": int(renewable_diagnostics["simulation_year"].iloc[0]),
}
pd.Series(summary, name="value")